### 기능 1 - 정제 데이터 불러오기 (load_clean_data)

In [1]:
import pandas as pd
import os
import numpy as np

# 파일 경로
BASE_DIR = os.getcwd()
FILE_PATH = os.path.join(os.path.dirname(BASE_DIR),
                         "week 4","data","student_habits_clean.csv")

# 데이터 불러오기
def load_clean_data(file_path):
    if not os.path.exists(file_path):
        print(f"과제 2를 먼저 실행하세요: {file_path}")
        print("현재 작업 폴더:", os.getcwd())
        print("찾는 파일 절대경로:", os.path.abspath(FILE_PATH))
        print("파일 존재 여부:", os.path.exists(FILE_PATH))
        return None

    df = pd.read_csv(file_path, encoding="utf-8-sig")

    print(f"데이터 로드 완료: {df.shape[0]}행 X {df.shape[1]}열")
    return df

### 기능 2 - 피처 선택 + 학습/테스트 데이터 분리 + 스케일링 (split_and_scale)

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 피처 선택
def split_and_scale(df):
    feature_cols = ["sleep_hours", "study_hours", "phone_hours",
                    "exercise_hours", "productive_hours", "sleep_sufficient"]

    X = df[feature_cols] # 입력 변수(X)
    y = df['score'] # 목표 변수(y)

    # 2. 학습/테스트 데이터 분리
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f'학습 데이터: {X_tr.shape}, 테스트 데이터: {X_val.shape}')

    # 3. 스케일링
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)
    
    return X_tr, X_val, y_tr, y_val, feature_cols 


### 기능 3 - 모델 학습 및 성능 평가 (train_and_evaluate)

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def train_and_evaluate(X_tr, X_val, y_tr, y_val):
    
    # 선형 회귀 모델 생성
    lr = LinearRegression()
    lr.fit(X_tr, y_tr)
    pred = lr.predict(X_val)

    # 예측값 평가
    mae = round(mean_absolute_error(y_val, pred), 3)
    rmse = round(root_mean_squared_error(y_val, pred), 3)
    r2 = round(r2_score(y_val, pred), 3)

    print(f'MAE: {mae} | RMSE: {rmse} | R2 : {r2}')

    return lr, pred

### 기능 4 - 실제 vs 예측 그래프 (plot_prediction)

In [4]:

import matplotlib
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

matplotlib.rcParams["font.family"] = "Malgun Gothic"   # Windows
# matplotlib.rcParams["font.family"] = "AppleGothic"   # macOS
matplotlib.rcParams["axes.unicode_minus"] = False       # 마이너스 부호 깨짐 방지

def plot_prediction(y_val, pred):
    
    plt.scatter(y_val, pred) # 산점도

    # 예측 기준선 그래프
    # 실제값과 예측값 전체 범위를 기준으로 최솟값·최댓값 계산
    min_value = min(np.min(y_val), np.min(pred))
    max_value = max(np.max(y_val), np.max(pred))

    # 완벽한 예측 기준선 (y = x)
    plt.plot([min_value, max_value], # x 좌표
             [min_value, max_value], # y 좌표
             linestyle = '--', color = 'red'
             )

    plt.title('실제 vs 예측 그래프')
    plt.xlabel('실제 점수')
    plt.ylabel('예측 점수')
    
    plt.tight_layout()

    plt.savefig('outputs/model_evaluation.png')
    
    plt.close()

### 기능 5 - 피처별 회귀 계수 출력 (show_coefficients)

In [5]:
def show_coefficient(lr, feature_cols):
    df_coef = pd.DataFrame({'features': feature_cols, 
                            'coefficient': lr.coef_.round(3)})

    df_coef = df_coef.sort_values('coefficient', ascending = False) # 회귀계수 내림차순 정렬

    print('\n[피처별 회귀 계수]\n', df_coef)
    
    return df_coef

### 기능 6 - main() 함수로 전체 연결

In [6]:
def main():
    df = load_clean_data(FILE_PATH)
    
    X_tr, X_val, y_tr, y_val, feature_cols = split_and_scale(df)
    lr, pred = train_and_evaluate(X_tr, X_val, y_tr, y_val)
    plot_prediction(y_val, pred)
    show_coefficient(lr, feature_cols)

if __name__ == "__main__":
    main()

데이터 로드 완료: 200행 X 13열
학습 데이터: (160, 6), 테스트 데이터: (40, 6)
MAE: 4.857 | RMSE: 6.595 | R2 : 0.531

[피처별 회귀 계수]
            features  coefficient
1       study_hours        3.795
4  productive_hours        3.645
0       sleep_hours        0.953
3    exercise_hours        0.596
5  sleep_sufficient        0.206
2       phone_hours       -2.896
